In [1]:
from peft import LoraConfig, TaskType, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from datasets import Dataset
from trl import SFTTrainer, SFTConfig, DPOTrainer, DPOConfig
from prompt import SYSTEM_PROMPT_CODE, SYSTEM_PROMPT_EVAL
import torch
import gc
import evaluate
from tqdm import tqdm
import re
import numpy as np
import os
from sentence_transformers import SentenceTransformer, util

/etc/python/sitecustomize.py:236: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


# Helper Functions

In [2]:
def parse_manual_function_call(text):
    """
    Manually parse JSON-formatted function calls from LLM output.
    Returns: (function_name, function_args) or None
    """
    found_functions = []

    # Remove <think> tag content
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # Find JSON-formatted function calls
    # Pattern matches: {"function": "xxx", "args": {...}}
    json_pattern = r'\{["\']function["\']\s*:\s*["\'](\w+)["\']\s*,\s*["\']args["\']\s*:\s*(\{[^}]*\})\s*\}'
    matches = re.findall(json_pattern, text)
    for match in matches:
        function_name = match[0]
        args_str = match[1]
        found_functions.append((function_name, args_str))

    # Also try matching JSON in code blocks
    code_block_pattern = r'```json\s*\n\s*\{["\']function["\']\s*:\s*["\'](\w+)["\']\s*,\s*["\']args["\']\s*:\s*(\{[^}]*\})\s*\}\s*\n\s*```'
    matches = re.findall(code_block_pattern, text, re.DOTALL)
    for match in matches:
        function_name = match[0]
        args_str = match[1]
        if (function_name, args_str) not in found_functions:
            found_functions.append((function_name, args_str))

    return found_functions

In [3]:
def evaluate_sft_model(model_id, model_dir, eval_dataset, output_dir, include_accuracy=True):
    # Load the fine-tuned model and tokenizer
    base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto")
    fine_tuned_model = PeftModel.from_pretrained(base_model, model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    eval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    # Define evaluation metrics
    metrics = evaluate.combine(['bleu', 'rouge', 'meteor'])
    bertscore = evaluate.load('bertscore')
    semantic_similarities = []
    if include_accuracy:
        accuracies = []

    for instance in tqdm(eval_dataset):
        # Tokenize input
        text = tokenizer.apply_chat_template(
            instance["messages"][:-1], add_generation_prompt=True, tokenize=False
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

        # Generate response
        generated_ids = fine_tuned_model.generate(
            **model_inputs,
            max_new_tokens=2048
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

        # Decode and extract model response
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        
        # Get the ground truth response
        gt_text = instance['messages'][-1]['content']
        
        # Compute semantic similarity
        gt_embedding = eval_model.encode(gt_text, convert_to_tensor=True)
        generated_embedding = eval_model.encode(generated_text, convert_to_tensor=True)

        # Compute metrics against the reference response
        metrics.add(prediction=generated_text, reference=gt_text)
        bertscore.add(prediction=generated_text, reference=gt_text)
        semantic_similarities.append(util.pytorch_cos_sim(gt_embedding, generated_embedding).item())
        
        # Compute parse accuracy
        if include_accuracy:
            generated_functions = parse_manual_function_call(generated_text)
            gt_functions = parse_manual_function_call(gt_text)
            accuracies.append(generated_functions == gt_functions)
    
    # Save metrics to output directory
    with open(output_dir, 'w') as file:
        mr = metrics.compute()
        bertscores = bertscore.compute(lang='en')
        bertscores.pop('hashcode')  # Remove hashcode from results
        bertscores = {f'bertscore_{k}': np.mean(v) for k, v in bertscores.items()}
        mr.update(bertscores)
        mr['semantic_similarity'] = np.mean(semantic_similarities)
        if include_accuracy:
            mr['accuracy'] = np.mean(accuracies)
        json.dump(mr, file, indent=4)
    
    # Clean up
    del base_model
    del fine_tuned_model
    del tokenizer
    del eval_model
    torch.cuda.empty_cache()
    gc.collect()

In [4]:
def evaluate_dpo_model(model_id, model_dir, eval_dataset, output_dir, include_accuracy=True):
    # Load the fine-tuned model and tokenizer
    base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto")
    fine_tuned_model = PeftModel.from_pretrained(base_model, model_dir)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    eval_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    
    # Define evaluation metrics
    metrics = evaluate.combine(['bleu', 'rouge', 'meteor'])
    bertscore = evaluate.load('bertscore')
    semantic_similarities = []
    if include_accuracy:
        accuracies = []

    for instance in tqdm(eval_dataset):
        # Tokenize input
        text = tokenizer.apply_chat_template(
            instance["prompt"], add_generation_prompt=True, tokenize=False
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

        # Generate response
        generated_ids = fine_tuned_model.generate(
            **model_inputs,
            max_new_tokens=2048
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

        # Decode and extract model response
        generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
        
        # Get the ground truth response
        gt_text = instance['chosen'][0]['content']
        
        # Compute semantic similarity
        gt_embedding = eval_model.encode(gt_text, convert_to_tensor=True)
        generated_embedding = eval_model.encode(generated_text, convert_to_tensor=True)

        # Compute metrics against the reference response
        metrics.add(prediction=generated_text, reference=gt_text)
        bertscore.add(prediction=generated_text, reference=gt_text)
        semantic_similarities.append(util.pytorch_cos_sim(gt_embedding, generated_embedding).item())
        
        # Compute parse accuracy
        if include_accuracy:
            generated_functions = parse_manual_function_call(generated_text)
            gt_functions = parse_manual_function_call(gt_text)
            accuracies.append(generated_functions == gt_functions)
    
    # Save metrics to output directory
    with open(output_dir, 'w') as file:
        mr = metrics.compute()
        bertscores = bertscore.compute(lang='en')
        bertscores.pop('hashcode')  # Remove hashcode from results
        bertscores = {f'bertscore_{k}': np.mean(v) for k, v in bertscores.items()}
        mr.update(bertscores)
        mr['semantic_similarity'] = np.mean(semantic_similarities)
        if include_accuracy:
            mr['accuracy'] = np.mean(accuracies)
        json.dump(mr, file, indent=4)
    
    # Clean up
    del base_model
    del fine_tuned_model
    del tokenizer
    del eval_model
    torch.cuda.empty_cache()
    gc.collect()

# SFT

## Prepare data

In [71]:
def preprocess_function(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_CODE},
            {"role": "user", "content": example['prompt']},
            {"role": "assistant", "content": example['correct']}
        ]
    }

In [72]:
with open("data/generated_data.json", "r") as f:
    data = json.load(f)
dataset = Dataset.from_list(data).map(preprocess_function, remove_columns=["prompt", "correct", "incorrect"]).train_test_split(test_size=0.2, seed=42)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 27587.21 examples/s]


In [73]:
dataset["train"][0]

{'messages': [{'content': 'You control a Kinova Gen3 robotic arm in Unity. Be concise and direct.\nThink step by step about the functions you need to call and the arguments they require to fully complete the user\'s request.\nEnsure that you are calling all functions necessary in the right order to achieve the desired outcome.\nThe current state of the simulation, including the names, positions, and rotations of all objects, is provided in JSON form in your most recent assistant message.\n\n## ⚠️ CRITICAL: Coordinate System\nUnity uses: **X = left/right, Y = UP/DOWN (vertical), Z = forward/back**\n- Move UP → increase Y (y > 0)\n- Move DOWN → decrease Y (y < 0)\n- Move LEFT → decrease X (x < 0)\n- Move RIGHT → increase X (x > 0)\n- Move FORWARD → increase Z (z > 0)\n- Move BACKWARD → decrease Z (z < 0)\n\n## CRITICAL: Function Call Format\n\nWhen you need to call a function, output ONLY this JSON format (nothing else):\n```json\n{"function": "function_name", "args": {"param": "value"}}

## Train model

In [74]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
output_dir = "coder_model_sft"

In [75]:
peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=32)

In [76]:
training_args = SFTConfig(
    # Training schedule / optimization
    per_device_train_batch_size = 4,      # Batch size per GPU
    # gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    # max_steps = 100,
    learning_rate = 1e-4,                 # Learning rate for the optimizer
    optim = "adamw_torch",           # Optimizer

    # Logging / reporting
    # logging_steps=1,                      # Log training metrics every N steps
    # report_to="trackio",                  # Experiment tracking tool
    # trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir='training_checkpoints',                     # Where to save model checkpoints and logs

    max_length=2048,                      # Maximum input sequence length
    # use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training
    activation_offloading=True,           # Offload activations to CPU to reduce GPU memory usage

    # Hub integration
    # push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`

)

In [77]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=peft_config
)

Truncating train dataset: 100%|██████████| 800/800 [00:00<00:00, 89378.38 examples/s]


In [78]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.945347
20,0.825022
30,0.675772
40,0.521214
50,0.330904
60,0.175375
70,0.105746
80,0.084750
90,0.078694
100,0.067594


TrainOutput(global_step=200, training_loss=0.21961687862873078, metrics={'train_runtime': 422.1306, 'train_samples_per_second': 1.895, 'train_steps_per_second': 0.474, 'total_flos': 5.493097165387776e+16, 'train_loss': 0.21961687862873078})

## Save Model

In [80]:
trainer.save_model(output_dir)

In [81]:
if 'model' in globals():
    del globals()['model']
if 'trainer' in globals():
    del globals()['trainer']
torch.cuda.empty_cache()
gc.collect()

15276

## Evaluate Model

In [82]:
evaluate_sft_model(model_id, output_dir, dataset["test"], "results/coder_sft.json")

Loading weights: 100%|██████████| 339/339 [00:01<00:00, 181.28it/s, Materializing param=model.norm.weight]                              
[nltk_data] Downloading package wordnet to /home/jmwood23/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jmwood23/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jmwood23/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
100%|██████████| 200/200 [11:52<00:00,  3.56s/it]


# DPO

## Prepare data

In [19]:
def preprocess_function(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_CODE},
            {"role": "user", "content": example['prompt']}
        ],
        "chosen": [{"role": "assistant", "content": example['correct']}],
        "rejected": [{"role": "assistant", "content": example['incorrect']}]
    }

In [ ]:
with open("data/generated_data.json", "r") as f:
    data = json.load(f)
dataset = Dataset.from_list(data).map(preprocess_function, remove_columns=["correct", "incorrect"]).train_test_split(test_size=0.2, seed=42)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 18297.60 examples/s]


In [21]:
dataset["train"][0]

{'prompt': [{'content': 'You control a Kinova Gen3 robotic arm in Unity. Be concise and direct.\nThink step by step about the functions you need to call and the arguments they require to fully complete the user\'s request.\nEnsure that you are calling all functions necessary in the right order to achieve the desired outcome.\nThe current state of the simulation, including the names, positions, and rotations of all objects, is provided in JSON form in your most recent assistant message.\n\n## ⚠️ CRITICAL: Coordinate System\nUnity uses: **X = left/right, Y = UP/DOWN (vertical), Z = forward/back**\n- Move UP → increase Y (y > 0)\n- Move DOWN → decrease Y (y < 0)\n- Move LEFT → decrease X (x < 0)\n- Move RIGHT → increase X (x > 0)\n- Move FORWARD → increase Z (z > 0)\n- Move BACKWARD → decrease Z (z < 0)\n\n## CRITICAL: Function Call Format\n\nWhen you need to call a function, output ONLY this JSON format (nothing else):\n```json\n{"function": "function_name", "args": {"param": "value"}}\n

## Train model

In [22]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
output_dir = "coder_model_dpo"

In [23]:
peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=32)

In [24]:
training_args = DPOConfig(
    beta=0.1,
    # Training schedule / optimization
    per_device_train_batch_size = 4,      # Batch size per GPU
    # gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    # max_steps = 100,
    learning_rate = 1e-4,                 # Learning rate for the optimizer
    optim = "adamw_torch",           # Optimizer

    # Logging / reporting
    # logging_steps=1,                      # Log training metrics every N steps
    # report_to="trackio",                  # Experiment tracking tool
    # trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir='training_checkpoints',                     # Where to save model checkpoints and logs

    max_length=2048,                      # Maximum input sequence length
    # use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training

    # Hub integration
    # push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`
)

In [25]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=peft_config
)

Tokenizing train dataset: 100%|██████████| 800/800 [00:01<00:00, 423.94 examples/s]


In [26]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.687379
20,0.618984
30,0.448348
40,0.238331
50,0.162641
60,0.178585
70,0.235128
80,0.162393
90,0.076871
100,0.192569


TrainOutput(global_step=200, training_loss=0.22019098252058028, metrics={'train_runtime': 347.2911, 'train_samples_per_second': 2.304, 'train_steps_per_second': 0.576, 'total_flos': 0.0, 'train_loss': 0.22019098252058028, 'epoch': 1.0})

## Save Model

In [27]:
trainer.save_model(output_dir)

In [28]:
if 'model' in globals():
    del globals()['model']
if 'trainer' in globals():
    del globals()['trainer']
torch.cuda.empty_cache()
gc.collect()

15043

## Evaluate Model

In [32]:
evaluate_dpo_model(model_id, output_dir, dataset["test"], "results/coder_dpo.json")

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 169.34it/s, Materializing param=model.norm.weight]                              
[nltk_data] Downloading package wordnet to /home/jmwood23/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jmwood23/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jmwood23/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
100%|██████████| 200/200 [12:42<00:00,  3.81s/it]


# Model inference

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading weights: 100%|██████████| 339/339 [00:01<00:00, 175.73it/s, Materializing param=model.norm.weight]                              


In [14]:
messages = [
  {
      'content': SYSTEM_PROMPT_CODE,
      'role': 'system',
  },
  {
      'content': "Move the gripper down and to the right by 20cm, down and to the left by 20cm, up and to the left by 20cm, and up and to the right by 20cm.",
      'role': 'user',
  }
]

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=2048
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

In [16]:
fine_tuned_model = PeftModel.from_pretrained(base_model, output_dir)

In [17]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

```json
{"function": "move_to_position", "args": {"x": 0, "y": -0.2, "z": 0, "relative": true}} // Move DOWN 20cm (Y-axis!)
{"function": "move_to_position", "args": {"x": 0.2, "y": 0, "z": 0, "relative": true}} // Move RIGHT 20cm (X-axis)
{"function": "move_to_position", "args": {"x": 0, "y": -0.2, "z": 0, "relative": true}} // Move DOWN 20cm (Y-axis!)
{"function": "move_to_position", "args": {"x": -0.2, "y": 0, "z": 0, "relative": true}} // Move LEFT 20cm (X-axis)
{"function": "move_to_position", "args": {"x": 0, "y": 0.2, "z": 0, "relative": true}}  // Move UP 20cm (Y-axis!)
{"function": "move_to_position", "args": {"x": -0.2, "y": 0, "z": 0, "relative": true}} // Move LEFT 20cm (X-axis)
{"function": "move_to_position", "args": {"x": 0, "y": 0.2, "z": 0, "relative": true}}  // Move UP 20cm (Y-axis!)
{"function": "move_to_position", "args": {"x": 0.2, "y": 0, "z": 0, "relative": true}} // Move RIGHT 20cm (X-axis)
```


# Evaluator SFT

## Prepare data

In [10]:
def preprocess_function(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_EVAL},
            {"role": "user", "content": example['prompt']},
            {"role": "assistant", "content": example['better']}
        ]
    }

In [11]:
with open("data/generated_data.json", "r") as f:
    data = json.load(f)
with open("data/generated_responses.json", "r") as f:
    responses = json.load(f)
    
for i, r in enumerate(responses):
    r["prompt"] = f"User Request: {data[i]['prompt']}\nCode: {data[i]['incorrect']}"
dataset = Dataset.from_list(responses).map(preprocess_function, remove_columns=["bad", "good", "better", "prompt"]).train_test_split(test_size=0.2, seed=42)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 24456.16 examples/s]


In [12]:
dataset["train"][0]

{'messages': [{'content': 'You are an agent evaluating the functional correctness of robot simulation code. Be concise and direct.\nEnsure that all functions necessary to achieve the user\'s request are present and being called in the correct order.\nAlso ensure that the correct arguments to fulfill the user\'s request are being passed into functions.\nMake sure that the direction for movement-based functions is correct as well.\nIf the necessary functions are present, the order of the code matches the user\'s request, and the correct arguments are being passed to each function, output \'True\' and nothing else. \nOtherwise, output \'False\', state the errors in the code, and provide suggestions for fixing the function calls and/or arguments.\nThe current state of the simulation, including the names, positions, and rotations of all objects, is provided in JSON form in your most recent assistant message.\n\n# ⚠️ CRITICAL: Coordinate System\nUnity uses: **X = left/right, Y = UP/DOWN (ver

## Train model

In [13]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
output_dir = "eval_model_sft"

In [14]:
peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=32)

In [15]:
training_args = SFTConfig(
    # Training schedule / optimization
    per_device_train_batch_size = 4,      # Batch size per GPU
    # gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    # max_steps = 100,
    learning_rate = 1e-4,                 # Learning rate for the optimizer
    optim = "adamw_torch",           # Optimizer

    # Logging / reporting
    # logging_steps=1,                      # Log training metrics every N steps
    # report_to="trackio",                  # Experiment tracking tool
    # trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir='training_checkpoints',                     # Where to save model checkpoints and logs

    max_length=2048,                      # Maximum input sequence length
    # use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training
    activation_offloading=True,           # Offload activations to CPU to reduce GPU memory usage

    # Hub integration
    # push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`

)

In [16]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=peft_config
)

Truncating train dataset: 100%|██████████| 800/800 [00:00<00:00, 83810.65 examples/s]
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.982278
20,0.880382
30,0.777123
40,0.654994
50,0.520753
60,0.345444
70,0.230258
80,0.192152
90,0.185930
100,0.153893


TrainOutput(global_step=200, training_loss=0.3228657430410385, metrics={'train_runtime': 428.934, 'train_samples_per_second': 1.865, 'train_steps_per_second': 0.466, 'total_flos': 5.676986080002048e+16, 'train_loss': 0.3228657430410385})

## Save model

In [18]:
trainer.save_model(output_dir)

In [19]:
if 'model' in globals():
    del globals()['model']
if 'trainer' in globals():
    del globals()['trainer']
torch.cuda.empty_cache()
gc.collect()

15276

## Evaluate model

In [20]:
evaluate_sft_model(model_id, output_dir, dataset["test"], "results/eval_sft.json", include_accuracy=False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 174.49it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[nltk_data] Downloading package wordnet to /home/jmwood23/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jmwood23/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jmwood23/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Loading weights: 100%|██████████| 389/389 [00:03<00:00, 124.08it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel L

# Evaluator DPO

## Prepare data

In [24]:
def preprocess_function(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_EVAL},
            {"role": "user", "content": example['prompt']}
        ],
        "chosen": [{"role": "assistant", "content": example['good']}],
        "rejected": [{"role": "assistant", "content": example['bad']}]
    }

In [27]:
with open("data/generated_data.json", "r") as f:
    data = json.load(f)
with open("data/generated_responses.json", "r") as f:
    responses = json.load(f)
    
for i, r in enumerate(responses):
    r["prompt"] = f"User Request: {data[i]['prompt']}\nCode: {data[i]['incorrect']}"
dataset = Dataset.from_list(responses).map(preprocess_function, remove_columns=["bad", "good", "better"]).train_test_split(test_size=0.2, seed=42)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 18047.70 examples/s]


In [28]:
dataset['train'][0]

{'prompt': [{'content': 'You are an agent evaluating the functional correctness of robot simulation code. Be concise and direct.\nEnsure that all functions necessary to achieve the user\'s request are present and being called in the correct order.\nAlso ensure that the correct arguments to fulfill the user\'s request are being passed into functions.\nMake sure that the direction for movement-based functions is correct as well.\nIf the necessary functions are present, the order of the code matches the user\'s request, and the correct arguments are being passed to each function, output \'True\' and nothing else. \nOtherwise, output \'False\', state the errors in the code, and provide suggestions for fixing the function calls and/or arguments.\nThe current state of the simulation, including the names, positions, and rotations of all objects, is provided in JSON form in your most recent assistant message.\n\n# ⚠️ CRITICAL: Coordinate System\nUnity uses: **X = left/right, Y = UP/DOWN (verti

## Train model

In [29]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
output_dir = "eval_model_dpo"

In [11]:
peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=32)

In [12]:
training_args = DPOConfig(
    beta=0.1,
    # Training schedule / optimization
    per_device_train_batch_size = 4,      # Batch size per GPU
    # gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    # max_steps = 100,
    learning_rate = 1e-4,                 # Learning rate for the optimizer
    optim = "adamw_torch",           # Optimizer

    # Logging / reporting
    # logging_steps=1,                      # Log training metrics every N steps
    # report_to="trackio",                  # Experiment tracking tool
    # trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir='training_checkpoints',                     # Where to save model checkpoints and logs

    max_length=2048,                      # Maximum input sequence length
    # use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training

    # Hub integration
    # push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`
)

In [14]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    peft_config=peft_config
)

Tokenizing train dataset: 100%|██████████| 800/800 [00:01<00:00, 440.22 examples/s]
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.617361
20,0.296801
30,0.139550
40,0.074747
50,0.139635
60,0.131549
70,0.093059
80,0.170297
90,0.047367
100,0.099728


TrainOutput(global_step=200, training_loss=0.12881084129214287, metrics={'train_runtime': 500.5325, 'train_samples_per_second': 1.598, 'train_steps_per_second': 0.4, 'total_flos': 0.0, 'train_loss': 0.12881084129214287, 'epoch': 1.0})

## Save Model

In [16]:
trainer.save_model(output_dir)

In [17]:
if 'model' in globals():
    del globals()['model']
if 'trainer' in globals():
    del globals()['trainer']
torch.cuda.empty_cache()
gc.collect()

15043

## Evaluate Model

In [30]:
evaluate_dpo_model(model_id, output_dir, dataset["test"], "results/eval_dpo.json", False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1311.89it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[nltk_data] Downloading package wordnet to /home/jmwood23/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jmwood23/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jmwood23/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1242.16it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaMode

# Evaluator DPO (GPT Coder-Evaluator Responses)

## Prepare data

In [4]:
def preprocess_function(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT_EVAL},
            {"role": "user", "content": example['prompt']}
        ],
        "chosen": [{"role": "assistant", "content": example['better']}],
        "rejected": [{"role": "assistant", "content": example['bad']}]
    }

In [22]:
with open("data/generated_data.json", "r") as f:
    data = json.load(f)
with open("data/generated_responses.json", "r") as f:
    responses = json.load(f)
    
for i, r in enumerate(responses):
    r["prompt"] = f"User Request: {data[i]['prompt']}\nCode: {data[i]['incorrect']}"
dataset = Dataset.from_list(responses).map(preprocess_function, remove_columns=["bad", "good", "better"]).train_test_split(test_size=0.2, seed=42)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 14801.56 examples/s]


In [9]:
dataset['train'][0]

{'prompt': [{'content': 'You are an agent evaluating the functional correctness of robot simulation code. Be concise and direct.\nEnsure that all functions necessary to achieve the user\'s request are present and being called in the correct order.\nAlso ensure that the correct arguments to fulfill the user\'s request are being passed into functions.\nMake sure that the direction for movement-based functions is correct as well.\nIf the necessary functions are present, the order of the code matches the user\'s request, and the correct arguments are being passed to each function, output \'True\' and nothing else. \nOtherwise, output \'False\', state the errors in the code, and provide suggestions for fixing the function calls and/or arguments.\nThe current state of the simulation, including the names, positions, and rotations of all objects, is provided in JSON form in your most recent assistant message.\n\n# ⚠️ CRITICAL: Coordinate System\nUnity uses: **X = left/right, Y = UP/DOWN (verti

## Train model

In [6]:
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
output_dir = "eval_model_dpo_gpt"

In [12]:
peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=32)

In [13]:
training_args = DPOConfig(
    beta=0.1,
    # Training schedule / optimization
    per_device_train_batch_size = 4,      # Batch size per GPU
    # gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    # max_steps = 100,
    learning_rate = 1e-4,                 # Learning rate for the optimizer
    optim = "adamw_torch",           # Optimizer

    # Logging / reporting
    # logging_steps=1,                      # Log training metrics every N steps
    # report_to="trackio",                  # Experiment tracking tool
    # trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir='training_checkpoints',                     # Where to save model checkpoints and logs

    max_length=2048,                      # Maximum input sequence length
    # use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training

    # Hub integration
    # push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`
)

In [14]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    peft_config=peft_config
)

Tokenizing train dataset: 100%|██████████| 800/800 [00:01<00:00, 432.75 examples/s]
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.590773
20,0.193724
30,0.100865
40,0.081700
50,0.129172
60,0.094543
70,0.104214
80,0.074381
90,0.044801
100,0.110043


TrainOutput(global_step=200, training_loss=0.11192882403731347, metrics={'train_runtime': 481.1326, 'train_samples_per_second': 1.663, 'train_steps_per_second': 0.416, 'total_flos': 0.0, 'train_loss': 0.11192882403731347, 'epoch': 1.0})

## Save model

In [16]:
trainer.save_model(output_dir)

In [17]:
if 'model' in globals():
    del globals()['model']
if 'trainer' in globals():
    del globals()['trainer']
torch.cuda.empty_cache()
gc.collect()

15035

## Evaluate model

In [23]:
evaluate_dpo_model(model_id, output_dir, dataset["test"], "results/eval_dpo_gpt.json", False)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1239.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[nltk_data] Downloading package wordnet to /home/jmwood23/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jmwood23/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jmwood23/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 913.91it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel